# Parkinson's Speech Detection System
### End-to-End Audio Feature Extraction and Machine Learning Pipeline
This notebook implements a complete machine learning system to detect Parkinson's Disease from voice recordings. The system is designed based on the principles of speech analysis, extracting features that reflect vocal instabilities characteristic of Parkinsonian dysphonia (e.g., vocal tremors, micro-instabilities in pitch and amplitude, and elevated breathiness/noise).

### Pipeline Stages:
1. **Requirements & Dependencies Setup**: Install and import the necessary libraries.
2. **Workspace & Folder Creation**: Dynamically create raw data, processed data, and model folders, and download the official dataset zip from the UCI repository.
3. **Audio Feature Extraction Pipeline**: Programmatically generate synthetic speech signals matching healthy and Parkinsonian clinical profiles to test our custom `librosa` acoustic feature extraction pipeline.
4. **Data Preprocessing & Train-Test Split**: Prepare the extracted features for machine learning models and partition into stratified training and testing sets.
5. **Model Training**: Train an XGBoost Classifier and a Random Forest Classifier comparison baseline.
6. **Model Evaluation & Visualizations**: Measure classifier performance and plot confusion matrix heatmaps.

---  
## Cell 1: Requirements & Dependencies Setup
In this cell, we install the necessary audio processing, data manipulation, and machine learning packages. Then, we import all required modules.

In [ ]:
# Install required libraries
!pip install librosa numpy pandas scikit-learn xgboost matplotlib seaborn joblib soundfile scipy

# Import core packages
import os
import glob
import zipfile
import io
import urllib.request
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Audio processing and wave writing
import librosa
import librosa.display
import scipy.io.wavfile as wav

# Data science & visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning pipelines & metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import joblib

print("Requirements check completed and all libraries imported successfully!")

---  
## Cell 2: Workspace & Folder Creation + Dataset Download
Here, we programmatically create the required subdirectories (`data/raw_audio/0/`, `data/raw_audio/1/`, `data/processed/`, and `models/`). We then download the official dataset zip from the UCI Machine Learning Repository, unzip it, and extract the text-based feature files from the inner RAR archive.

In [ ]:
# 1. Create workspaces dynamically using Python's os module
dirs = [
    "data/raw_audio/0",  # 0 for Healthy/Control
    "data/raw_audio/1",  # 1 for Parkinson's Disease (PD)
    "data/processed",    # CSV with processed features
    "models"             # Saved joblib classifiers & scalers
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"Workspace directory verified: {d}")

# 2. Programmatically download dataset from UCI
uci_zip_url = "https://archive.ics.uci.edu/static/public/301/parkinson+speech+dataset+with+multiple+types+of+sound+recordings.zip"
zip_destination = "data/uci_dataset.zip"

print(f"\nDownloading UCI dataset ZIP from: {uci_zip_url}...")
try:
    headers = {'User-Agent': 'Mozilla/5.0'}
    req = urllib.request.Request(uci_zip_url, headers=headers)
    with urllib.request.urlopen(req) as response:
        with open(zip_destination, "wb") as f:
            f.write(response.read())
    print("UCI Dataset ZIP downloaded successfully!")
    
    # Extract ZIP contents
    with zipfile.ZipFile(zip_destination, 'r') as zip_ref:
        zip_ref.extractall("data")
    print("ZIP file extracted. Folder contents:", os.listdir("data"))
    
    # Extract raw Parkinson_Multiple_Sound_Recording.rar file using tar (built-in support in modern systems)
    rar_file = "data/Parkinson_Multiple_Sound_Recording.rar"
    if os.path.exists(rar_file):
        print(f"\nExtracting RAR archive '{rar_file}'...")
        subprocess.run(["tar", "-xf", rar_file, "-C", "data"], check=True)
        print("RAR archive extracted successfully! Files available:", 
              [f for f in os.listdir("data") if f.endswith(".txt")])
    else:
        print("Warning: RAR file not found.")
        
except Exception as e:
    print(f"Error fetching dataset: {e}")

---  
## Cell 3: Audio Feature Extraction Pipeline (Librosa & Synthetic Voice Generator)
Since the UCI dataset distributes tabular feature files (`train_data.txt` and `test_data.txt`) rather than raw audio files, we will create a high-quality **Synthetic Voice Generator** that models the clinical biomarkers of Parkinson's Disease. This allows us to construct and test a complete `librosa` audio feature extraction pipeline.

### Modeling Clinical Speech Biomarkers for Parkinson's:
- **Healthy Voice (0)**: Steady fundamental frequency (e.g., $F_0 = 150\text{ Hz}$) with high amplitude and frequency stability.
- **Parkinson's Voice (1)**:
  - **Vocal Tremor**: Low-frequency modulation ($4\text{ to }7\text{ Hz}$) of the pitch carrier wave.
  - **Jitter (frequency instability)**: Micro-fluctuations in phase cycles.
  - **Shimmer (amplitude instability)**: Micro-fluctuations in amplitude envelopes.
  - **Lower HNR (Harmonics-to-Noise Ratio)**: Introduced using additive white Gaussian noise to represent voice breathiness.

We will generate 20 Healthy samples and 20 Parkinson's samples in the raw audio folders, pass them through our feature extractor function extracting **20 MFCCs, Spectral Centroid, Spectral Rolloff, Spectral Bandwidth, Zero Crossing Rate (ZCR), Chroma STFT, and RMS Energy**, and output `data/processed/parkinsons_extracted_features.csv`.

In [ ]:
# 1. Helper to generate synthetic audio files based on clinical voice profiles
def generate_synthetic_voice(file_path, label, duration=1.5, sr=22050):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    f0 = 150.0  # fundamental frequency
    
    if label == 0:
        # Healthy: clean, stable sine wave
        signal = np.sin(2 * np.pi * f0 * t)
    else:
        # Parkinson's: introduce voice impairments
        # Vocal Tremor (frequency modulation around 6Hz)
        tremor_f = 6.0
        tremor = 5.0 * np.sin(2 * np.pi * tremor_f * t)
        
        # Jitter: micro frequency pitch perturbations
        jitter = 2.0 * np.sin(2 * np.pi * 35.0 * t)
        
        # Phase calculation incorporating frequency instabilities
        phase = 2 * np.pi * (f0 * t + tremor * t + jitter * t)
        
        # Shimmer: amplitude instability
        shimmer = 1.0 + 0.15 * np.sin(2 * np.pi * 8.0 * t)
        
        # Signal with tremor, jitter, and shimmer
        signal = shimmer * np.sin(phase)
        
        # Add white noise (lowering Harmonics-to-Noise Ratio)
        noise = np.random.normal(0, 0.1, len(signal))
        signal = signal + noise
        
    # Normalize volume
    signal = signal / np.max(np.abs(signal))
    
    # Convert to 16-bit PCM integer WAV format and save
    signal_int16 = (signal * 32767).astype(np.int16)
    wav.write(file_path, sr, signal_int16)

# 2. Check and generate synthetic audio files if directories are empty
healthy_dir = "data/raw_audio/0"
pd_dir = "data/raw_audio/1"
num_samples = 20

existing_healthy = glob.glob(os.path.join(healthy_dir, "*.wav"))
if len(existing_healthy) == 0:
    print(f"Generating {num_samples} Healthy WAV files in {healthy_dir}...")
    for i in range(num_samples):
        generate_synthetic_voice(os.path.join(healthy_dir, f"healthy_{i+1}.wav"), label=0)
else:
    print(f"Found {len(existing_healthy)} existing Healthy WAV files.")

existing_pd = glob.glob(os.path.join(pd_dir, "*.wav"))
if len(existing_pd) == 0:
    print(f"Generating {num_samples} Parkinson's WAV files in {pd_dir}...")
    for i in range(num_samples):
        generate_synthetic_voice(os.path.join(pd_dir, f"parkinson_{i+1}.wav"), label=1)
else:
    print(f"Found {len(existing_pd)} existing Parkinson's WAV files.")

# 3. Centralized Feature Extraction using features.py
from features import extract_features_with_breakdown

feature_names = (
    [f"mfcc_{i+1}" for i in range(20)] + 
    ["spectral_centroid", "spectral_rolloff", "spectral_bandwidth", "zcr", "chroma", "rms"]
)

# Run extraction loop
print("\nRunning feature extraction on WAV signals...")
dataset_features = []
dataset_labels = []

for f in glob.glob(os.path.join(healthy_dir, "*.wav")):
    vector, _ = extract_features_with_breakdown(f)
    dataset_features.append(vector.flatten())
    dataset_labels.append(0)

for f in glob.glob(os.path.join(pd_dir, "*.wav")):
    vector, _ = extract_features_with_breakdown(f)
    dataset_features.append(vector.flatten())
    dataset_labels.append(1)

# Construct dataframe and save
df = pd.DataFrame(dataset_features, columns=feature_names)
df['status'] = dataset_labels

output_csv = "data/processed/parkinsons_extracted_features.csv"
df.to_csv(output_csv, index=False)
print(f"Extracted dataset saved successfully! Shape: {df.shape} -> Location: {output_csv}")
df.head()

---  
## Cell 4: Data Preprocessing & Train-Test Split
In this cell, we load the saved feature dataset, split the columns into features ($X$) and the target label ($y$, `status`), perform an 80/20 train-test split stratified on the target class, and apply feature standardization using `StandardScaler`.

In [ ]:
# 1. Load CSV file
processed_csv = "data/processed/parkinsons_extracted_features.csv"
data_df = pd.read_csv(processed_csv)

# 2. Separate features (X) and label (y)
X = data_df.drop(columns=['status'])
y = data_df['status']

# 3. Train-test split (80-20) with stratification by status target y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Testing set:  X_test = {X_test.shape}, y_test = {y_test.shape}")

# 4. Fit and apply StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFirst 5 scaled feature means (Target near 0):", np.mean(X_train_scaled, axis=0)[:5])
print("First 5 scaled feature standard deviations (Target 1):", np.std(X_train_scaled, axis=0)[:5])

---  
## Cell 5: Model Training (XGBoost & Random Forest)
We initialize and fit both XGBoost (`XGBClassifier`) and Random Forest (`RandomForestClassifier`) models. Once trained, we serialize both the XGBoost model and the fitted scaler into the `models/` folder using `joblib`.

In [ ]:
# 1. Initialize XGBoost and Random Forest classifiers
xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# 2. Train classifiers on the scaled training dataset
print("Training XGBoost classifier...")
xgb_clf.fit(X_train_scaled, y_train)

print("Training Random Forest classifier...")
rf_clf.fit(X_train_scaled, y_train)

# 3. Save XGBoost and fitted StandardScaler
xgb_model_path = "models/xgboost_parkinsons_model.joblib"
scaler_path = "models/standard_scaler.joblib"

joblib.dump(xgb_clf, xgb_model_path)
joblib.dump(scaler, scaler_path)

print(f"\nSaved XGBoost model to: {xgb_model_path}")
print(f"Saved StandardScaler to: {scaler_path}")

---  
## Cell 6: Model Evaluation & Visualizations
In this final cell, we run predictions on the test dataset using both classifiers. We calculate the performance metrics (Accuracy, Precision, Recall, F1-Score) and plot comparison Confusion Matrices using `seaborn` heatmaps.

In [ ]:
# 1. Predict labels on scaled test set
xgb_pred = xgb_clf.predict(X_test_scaled)
rf_pred = rf_clf.predict(X_test_scaled)

# 2. Evaluate performance
def print_classification_results(model_name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"==================== {model_name} ====================")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=['Healthy (0)', "Parkinson's (1)"]))

print_classification_results("XGBoost Classifier", y_test, xgb_pred)
print_classification_results("Random Forest Classifier (Baseline)", y_test, rf_pred)

# 3. Plot Confusion Matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = ['Healthy', "Parkinson's"]

# XGBoost Confusion Matrix
xgb_cm = confusion_matrix(y_test, xgb_pred)
sns.heatmap(xgb_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=classes, yticklabels=classes)
axes[0].set_title("XGBoost Classifier Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# Random Forest Confusion Matrix
rf_cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1], 
            xticklabels=classes, yticklabels=classes)
axes[1].set_title("Random Forest Classifier Confusion Matrix")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

plt.tight_layout()
plt.show()